## Incremental Data Ingestion with Auto Loader
- *Concept:* **Auto Loader** is a Databricks feature built on top of **Spark Structured Streaming** that incrementally and efficiently processes new data files as they arrive in cloud storage (ADLS Gen2, S3, GCS) without needing to manually track which files have already been loaded.
- *Why Auto Loader over Batch Reads?*
  - **Scalability:** Efficiently discovers millions of files using directory listing or file notification services.
  - **Exactly-Once Processing:** Automatically tracks processed files using **Checkpointing**, ensuring no duplicate data is ingested.
  - **Schema Inference & Evolution:** Automatically detects schemas and handles schema changes (new columns, data type mismatches) without breaking the pipeline.
  - **Format Identifier:** Auto Loader uses the `cloudFiles` source format in `spark.readStream`.

## Storage Setup (ADLS Gen2 Directories)
- Inside your existing ADLS Gen2 container (`mycontainer`), create two directories:
  - *Source Directory (`autosource`):* `abfss://mycontainer@storage_account_name.dfs.core.windows.net/autosource` — upload your initial `.parquet` file here.
  - *Sink Directory (`autosink`):* `abfss://mycontainer@storage_account_name.dfs.core.windows.net/autosink` — Auto Loader will automatically create two subfolders inside this:
    - `autosink/check`: Stores the schema location and streaming checkpoints.
    - `autosink/data`: Stores the incrementally ingested output data.

## Reading Data Incrementally with Auto Loader (PySpark)
- *Concept:* Use `spark.readStream.format("cloudFiles")` to read files incrementally from `autosource`. Because of Spark's **lazy evaluation**, running `readStream` alone only checks syntax and does not start ingesting data until `writeStream` is called.
- *Key Options:*
  - `cloudFiles.format`: Specifies the incoming raw file format (e.g., `parquet`, `csv`, `json`).
  - `cloudFiles.schemaLocation`: Path (`autosink/check`) where Auto Loader caches the inferred schema and updates it if the schema evolves.
- *PySpark Example:*
```python
from pyspark.sql.functions import *

df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "parquet")
    .option("cloudFiles.schemaLocation", "abfss://mycontainer@storage_account_name.dfs.core.windows.net/autosink/check")
    .load("abfss://mycontainer@storage_account_name.dfs.core.windows.net/autosource")
)
```

## Writing Incrementally (`writeStream`)
- *Concept:* Write the streaming DataFrame into `autosink/data` using `writeStream`.
- *Key Options:*
  - `checkpointLocation`: **Mandatory** (`autosink/check`). Tracks processed files using **RocksDB** so only new files are ingested.
  - `trigger`:
    - `processingTime='10 seconds'`: Live streaming—checks `autosource` every 10 seconds for new files.
    - `availableNow=True`: Batch-like trigger—processes all pending files once and automatically stops the stream.
- *PySpark Example:*
```python
(
    df.writeStream
    .format("delta")
    .option("checkpointLocation", "abfss://mycontainer@storage_account_name.dfs.core.windows.net/autosink/check")
    .trigger(processingTime='10 seconds')
    .start("abfss://mycontainer@storage_account_name.dfs.core.windows.net/autosink/data")
)
```

## Monitoring, Checkpoint Internals & Cleanup
- *Live Monitoring & QPL:* While the stream runs, Databricks displays a live ingestion graph and the **QPL (Query Progress Log)** under raw data (showing `batchId`, timestamp, and processing time).
- *Testing Incremental Load:* Upload a second `.parquet` file into `autosource`. The graph spikes automatically and copies **only the new file** into `autosink/data` (2 files total, zero duplicates).
- *Inside `autosink/check` (Checkpoint Folder):*
  - `metadata`, `offsets`, `commits`: Created by `writeStream` to track streaming state.
  - `sources` -> **RocksDB**: Created by `schemaLocation` / Auto Loader to store metadata logs of processed files and schema versions (`0`, `1`, etc.).
- *Important Cost Tip:* Always click **Cancel** on the streaming cell once testing is complete so the continuous stream does not keep your cluster running indefinitely.

## Spark Streaming DataFrame (Core Concepts)
- *Concept:* Spark Structured Streaming lets you process continuously arriving data using the same DataFrame API as batch processing—simply replace `spark.read` with `spark.readStream` and `df.write` with `df.writeStream`.
- *Lazy Evaluation:* Running `spark.readStream` only validates syntax and defines the streaming DataFrame; actual data ingestion does **not** begin until an action (`df.writeStream...start()`) is triggered.
- *Triggers (Batch vs. Live Streaming):*
  - `trigger(processingTime='10 seconds')`: Continuously checks the source directory every 10 seconds for new files (live micro-batch streaming).
  - `trigger(availableNow=True)`: Processes all pending files in the source once and automatically stops the stream (ideal for scheduled batch-style incremental loads).
- *Monitoring:* While running, Databricks displays a live ingestion line chart and the **QPL (Query Progress Log)** under raw data (tracking `batchId`, timestamp, and processing time).

## Databricks Workflows & Jobs (Orchestration)
- *Concept:* **Workflows** is Databricks' built-in orchestration service used to build and automate end-to-end data pipelines. It executes multiple tasks (notebooks, SQL queries, Python scripts) in a defined sequence as a **DAG (Directed Acyclic Graph)** (e.g., `Task 1` → `Task 2` → `Task 3`).

### Step 1: Create Sample Notebooks
- Create two notebooks in your workspace to simulate a multi-step pipeline:
  - *`notebook1`:*
```python
print("Hello from Notebook 1")
```
  - *`notebook2`:*
```python
print("Hello from Notebook 2")
```

### Step 2: Create a Workflow Job (UI Steps)
1. In the left sidebar of Databricks, click on **Workflows** → **Create job**.
2. **Configure Task 1:**
   - *Task name:* `task1`
   - *Type:* `Notebook`
   - *Source:* `Workspace` → Browse and select `notebook1`.
   - *Compute / Cluster:* Select your existing cluster (see Pro Tip below).
   - *(Optional)* Add **Parameters** (key-value pairs via UI or JSON) or **Dependent libraries** if needed.
   - Click **Create task** (or **Save and continue**).
3. **Configure Task 2 (Create a Dependency / DAG):**
   - Click **+ Add task** → select **Notebook**.
   - *Task name:* `task2`
   - *Path:* Select `notebook2`.
   - *Depends on:* Ensure `task1` is selected so `task2` only runs after `task1` succeeds.
   - *Compute / Cluster:* Select your existing cluster.
   - Click **Create task**.

### Step 3: Run & Monitor the Pipeline
- Click **Run now** at the top right, then click **View run** to monitor execution.
- In the **Runs** view, you can track:
  - **DAG Progress:** Visual status of each task running in sequence.
  - **Metadata:** `Job ID`, `Run ID`, `Start time`, `Duration`, and `Status` (Succeeded / Failed).
  - **Task Output:** Click on any completed task (`task1` or `task2`) to view its printed notebook output and logs.

### Pro Tip: Job Cluster vs. All-Purpose Cluster
- **Job Cluster (Default in Workflows):** Spins up a fresh cluster for the job and terminates automatically when finished. Best for **production** (lower cost), but takes a few minutes to start up.
- **All-Purpose (Existing) Cluster:** Runs immediately if your cluster is already active. Best for **quick hands-on testing and development**.